# LeetCode #1416: Restore The Array

https://leetcode.com/problems/restore-the-array/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n \cdot k)$ exponential | $O(n)$ |
| **Optimal: Prefix-Sum DP ★** | $O(n \log k)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Try every way to split the string into numbers $\leq k$ by recursively choosing cut points, with memoization. Without smart pruning each position scans back $O(\text{digits}(k))$ characters — effectively $O(n \log k)$ — but the naive recursive form is much worse.

### Optimal: Prefix-Sum DP ★
`dp[i]` = number of valid arrays formed from `s[0..i-1]`. For each position $i$, look back at all valid starting indices $j$ (no leading zeros, value $\leq k$) and accumulate `dp[j]`. The window of valid $j$ values is at most $O(\log_{10} k)$ wide (digits of $k$), so each position takes $O(\log k)$ work. All arithmetic is done modulo $10^9 + 7$.

**Constraints:**
* $1 \leq s.\text{length} \leq 10^6$
* $1 \leq k \leq 10^9$


## Solutions

### C#

In [ ]:
public class Solution {
    const int MOD = 1_000_000_007;

    public int NumberOfArrays(string s, int k) {
        int n = s.Length;
        // dp[i] = number of valid arrays using the first i characters
        long[] dp = new long[n + 1];
        dp[0] = 1; // Empty prefix has one valid (empty) array

        for (int i = 1; i <= n; i++) {
            long num = 0;
            // Walk back from position i; stop when the number exceeds k or has a leading zero
            for (int j = i - 1; j >= 0; j--) {
                // Build the number s[j..i-1] right-to-left
                num = (s[j] - '0') * (long)Math.Pow(10, i - 1 - j) + num;
                if (s[j] == '0') break;   // Leading zero: no valid split starting here
                if (num > k) break;        // Exceeds k: no point looking further left
                dp[i] = (dp[i] + dp[j]) % MOD;
            }
        }

        return (int)dp[n];
    }
}

### Python

In [ ]:
class Solution:
    def numberOfArrays(self, s: str, k: int) -> int:
        MOD = 10**9 + 7
        n = len(s)
        # dp[i] = number of valid arrays using the first i characters
        dp = [0] * (n + 1)
        dp[0] = 1  # Empty prefix has one valid (empty) array
        max_digits = len(str(k))  # At most this many digits can form a valid number

        for i in range(1, n + 1):
            # Walk back at most max_digits characters to stay within range k
            for j in range(max(0, i - max_digits), i):
                if s[j] == '0':
                    continue  # Leading zero: skip this starting index
                num = int(s[j:i])
                if num > k:
                    continue  # Exceeds k: not a valid segment
                dp[i] = (dp[i] + dp[j]) % MOD

        return dp[n]


### Go

In [ ]:
func numberOfArrays(s string, k int) int {
    const MOD = 1_000_000_007
    n := len(s)
    // dp[i] = number of valid arrays using the first i characters
    dp := make([]int, n+1)
    dp[0] = 1 // Empty prefix has one valid (empty) array

    // Compute max digits in k to bound the look-back window
    maxDigits := 0
    for tmp := k; tmp > 0; tmp /= 10 {
        maxDigits++
    }

    for i := 1; i <= n; i++ {
        num := 0
        for j := i - 1; j >= 0 && i-j <= maxDigits; j-- {
            if s[j] == '0' && j < i-1 {
                continue // Leading zero in multi-digit number
            }
            digit := int(s[j] - '0')
            // Prepend digit to the number built so far
            pow := 1
            for p := 0; p < i-1-j; p++ {
                pow *= 10
            }
            num = digit*pow + (num % pow * 10 / 10)
            _ = num
            // Simpler: parse the substring directly
            val := 0
            overflow := false
            for p := j; p < i; p++ {
                val = val*10 + int(s[p]-'0')
                if val > k {
                    overflow = true
                    break
                }
            }
            if s[j] == '0' {
                break // Leading zero: stop scanning
            }
            if !overflow && val <= k {
                dp[i] = (dp[i] + dp[j]) % MOD
            }
        }
    }

    return dp[n]
}

### Rust

In [ ]:
impl Solution {
    pub fn number_of_arrays(s: String, k: i32) -> i32 {
        const MOD: i64 = 1_000_000_007;
        let k = k as i64;
        let bytes = s.as_bytes();
        let n = bytes.len();

        // Compute max digits in k to bound the look-back window
        let max_digits = k.to_string().len();

        // dp[i] = number of valid arrays using the first i characters
        let mut dp = vec![0i64; n + 1];
        dp[0] = 1; // Empty prefix has one valid (empty) array

        for i in 1..=n {
            let start = if i > max_digits { i - max_digits } else { 0 };
            for j in start..i {
                if bytes[j] == b'0' {
                    continue; // Leading zero: skip this starting index
                }
                let num: i64 = s[j..i].parse().unwrap_or(k + 1);
                if num <= k {
                    // Segment s[j..i] is a valid number; inherit paths from dp[j]
                    dp[i] = (dp[i] + dp[j]) % MOD;
                }
            }
        }

        dp[n] as i32
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `s = "1000", k = 10000`
Valid splits: `[1,0,0,0]` — wait, 0 is not in $[1, k]$. Actually `s = "1000"` with leading-zero rules: `1000` itself is valid ($\leq 10000$), giving dp = 1. Only one valid array.

### 2. Slightly Complex
**Input:** `s = "1317", k = 2000`
Valid numbers $\leq 2000$: `1`, `13`, `131`, `1317`; and sub-segments. Multiple splits are valid — the DP accumulates their counts across positions.

### 3. Edge Case: Time Factor
**Input:** `s = "111...1"` ($10^6$ ones), `k = 10^9`
Each position looks back at most 9 digits (since $k < 10^{10}$), so the total work is $\approx 9 \times 10^6$ — the worst-case time path at maximum string length.

### 4. Edge Case: Space Factor
**Input:** `s = "123456789"`, `k = 1`
Only single-digit segments are valid, and only '1' fits ($\leq 1$). Almost no paths survive, but the DP array still allocates $n + 1 = 10$ entries — $O(n)$ space at its minimum practical size.

### 5. Almost-Impossible but Plausible
**Input:** `s` is $10^6$ copies of `'9'`, `k = 10^9`
The 9-digit window `999999999 < 10^9$ is valid but `9999999999` is not; so segments of length $\leq 9$ are counted. The modular accumulation prevents any overflow despite counts potentially reaching $10^{900000}$ without modulo.
